<a href="https://colab.research.google.com/github/hdev14/tech-challenger-03/blob/main/FIAP_TECH_CHALLENGER_FASE_3_MODEL_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
!pip install -q chromadb sentence-transformers
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
!pip install transformers datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-_3puau73/unsloth_5842b03b7b49479dbef2e0b02cca4375
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-_3puau73/unsloth_5842b03b7b49479dbef2e0b02cca4375
  Resolved https://github.com/unslothai/unsloth.git to commit de1f4a743409f0c8403947d6a94144b65dc67635
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully uninstalled trl-0.8.6
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully 

In [13]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = '/content/drive/MyDrive/FIAP/cancer_pre_trained_model',
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1551: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/FIAP/cancer_pre_trained_model as a legacy tokenizer.
Unsloth 2026.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128255)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [14]:
import chromadb

db_path = '/content/drive/MyDrive/FIAP/chroma_db'
client = chromadb.PersistentClient(path=db_path)

In [30]:
try:
    collection = client.get_collection(name='cancer_qa_collection')
except Exception as e:
    print(f"An error occurred while fetching or querying the collection: {e}")

In [16]:
questions = [
    "What are the key symptoms of pancreatic cancer?",
    "What is colon cancer and what affects its prognosis?",
    "How is renal cell cancer defined?",
    "Tell me about the Game of Thrones TV Show?"
]


prompt_template = """
You are a medical AI assistant. Answer the question using ONLY the provided context.
If the answer cannot be found in the context, say "I can't answer this question".

Context:
{context}

Question: {question}

Answer:"""

for question in questions:
    results = collection.query(
      query_texts=[question],
      n_results=1
    )

    retrieved_doc = results['documents'][0][0] if 'cancer' in question.lower() and results['documents'] and results['documents'][0] else "No context found."

    formatted_prompt = prompt_template.format(context=retrieved_doc, question=question)

    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.5, use_cache=True)
    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    generated_answer = decoded_output[len(formatted_prompt):].strip()

    print(f"=== Question: {question} ===")
    print(f"\nGenerated Answer:\n{generated_answer}\n")
    print(f"Source ({results['metadatas'][0][0]['file_source']}):\n{retrieved_doc[:300]}...")
    print("="*50 + "\n\n\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Question: What are the key symptoms of pancreatic cancer? ===

Generated Answer:
Signs and symptoms of pancreatic cancer include jaundice, pain, and weight loss. 
                    Pancreatic cancer may not cause early signs or symptoms. Signs and symptoms may be caused by pancreatic cancer or by other conditions. Check with your doctor if you have any of the following:         -  Jaundice (yellowing of the skin and whites of the eyes).    - Light-colored stools.    - Dark urine.    - Pain in the upper or middle abdomen and back.     - Weight loss for no known reason.     - Loss of appetite.    - Feeling very tired.
	        	
		        
		                Pancreatic cancer is difficult to detect (find) and diagnose early.
		                Pancreatic cancer is difficult to detect and diagnose

Source (0000025_1.xml):
Question: What are the symptoms of Pancreatic Cancer ?
Answer: Signs and symptoms of pancreatic cancer include jaundice, pain, and weight loss. 
	                   

Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Question: What is colon cancer and what affects its prognosis? ===

Generated Answer:
Certain factors affect prognosis (chance of recovery) and treatment options. The prognosis (chance of recovery) and treatment options depend on the following:         - The stage of the cancer (whether the cancer is in the inner lining of the colon only or has spread through the colon wall, or has spread to lymph nodes or other places in the body).    - Whether the cancer has blocked or made a hole in the colon.    - Whether there are any cancer cells left after surgery.    - Whether the cancer has recurred.     - The patients general health.        The prognosis also depends on the blood levels of carcinoembryonic antigen (CEA) before treatment begins. CEA is a substance in the blood that may be increased

Source (0000037_1.xml):
Question: What is the outlook for Colon Cancer ?
Answer: Certain factors affect prognosis (chance of recovery) and treatment options. The prognosis (chance of recovery) 

Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Question: How is renal cell cancer defined? ===

Generated Answer:
Renal cell cancer is a disease in which malignant (cancer) cells form in tubules of the kidney.    - Renal cell cancer is a disease in which malignant (cancer) cells form in tubules of the kidney.    - Smoking and misuse of certain pain medicines can affect the risk of renal cell cancer.     - Signs of renal cell cancer include blood in the urine and a lump in the abdomen.    - Tests that examine the abdomen and kidneys are used to detect (find) and diagnose renal cell cancer.    - Certain factors affect prognosis (chance of recovery) and treatment options.
                
                
                    Smoking and misuse of certain pain medicines can affect the risk of renal cell cancer.
                    Anything that increases your risk of getting a

Source (0000026_1.xml):
Question: What is (are) Renal Cell Cancer ?
Answer: Key Points
                    - Renal cell cancer is a disease in which maligna

In [17]:
!pip install -U langchain langchain-community langchain-chroma langchain-classic

In [26]:
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_core.language_models.llms import LLM
from typing import Any, List, Mapping, Optional
import torch

In [27]:
class CancerModelLLM(LLM):
    model: Any
    tokenizer: Any
    max_new_tokens: int = 150
    temperature: float = 0.5

    @property
    def _llm_type(self) -> str:
        return "cancer-model"

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        **kwargs: Any,
    ) -> str:
        inputs = self.tokenizer([prompt], return_tensors="pt").to("cuda")

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            use_cache=True
        )

        decoded_output = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

        return decoded_output.strip()

In [28]:
import chromadb
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vector_store = Chroma(
    client=client,
    collection_name="cancer_qa_collection",
    embedding_function=embeddings
)

retriever = vector_store.as_retriever(search_kwargs={"k": 1})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
cancer_llm = CancerModelLLM(model=model, tokenizer=tokenizer)

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

qa_chain = RetrievalQA.from_chain_type(
    llm=cancer_llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

for question in questions:
    try:
        response = qa_chain.run(question)
        print(f"{response}")
    except Exception as e:
        print(f"An error occurred: {e}")
    print("="*50 + "\n")

Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You are a medical AI assistant. Answer the question using ONLY the provided context.
If the answer cannot be found in the context, say "I can't answer this question".

Context:
Question: What are the symptoms of Pancreatic Cancer?
Answer: Signs and symptoms of pancreatic cancer include jaundice, pain, and weight loss. 
	                    Pancreatic cancer may not cause early signs or symptoms. Signs and symptoms may be caused by pancreatic cancer or by other conditions. Check with your doctor if you have any of the following:         -  Jaundice (yellowing of the skin and whites of the eyes).    - Light-colored stools.    - Dark urine.    - Pain in the upper or middle abdomen and back.     - Weight loss for no known reason.     - Loss of appetite.    - Feeling very tired.
	        	
		        
		                Pancreatic cancer is difficult to detect (find) and diagnose early.
		                Pancreatic cancer is difficult to detect and diagnose for the following reasons:         

Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You are a medical AI assistant. Answer the question using ONLY the provided context.
If the answer cannot be found in the context, say "I can't answer this question".

Context:
Question: What is the outlook for Colon Cancer?
Answer: Certain factors affect prognosis (chance of recovery) and treatment options. The prognosis (chance of recovery) and treatment options depend on the following:         - The stage of the cancer (whether the cancer is in the inner lining of the colon only or has spread through the colon wall, or has spread to lymph nodes or other places in the body).    - Whether the cancer has blocked or made a hole in the colon.    - Whether there are any cancer cells left after surgery.    - Whether the cancer has recurred.     - The patients general health.        The prognosis also depends on the blood levels of carcinoembryonic antigen (CEA) before treatment begins. CEA is a substance in the blood that may be increased when cancer is present.

Question: What is colon ca

Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You are a medical AI assistant. Answer the question using ONLY the provided context.
If the answer cannot be found in the context, say "I can't answer this question".

Context:
Question: What is (are) Renal Cell Cancer?
Answer: Key Points
                    - Renal cell cancer is a disease in which malignant (cancer) cells form in tubules of the kidney.    - Smoking and misuse of certain pain medicines can affect the risk of renal cell cancer.     - Signs of renal cell cancer include blood in the urine and a lump in the abdomen.    - Tests that examine the abdomen and kidneys are used to detect (find) and diagnose renal cell cancer.    - Certain factors affect prognosis (chance of recovery) and treatment options.
                
                
                    Renal cell cancer is a disease in which malignant (cancer) cells form in tubules of the kidney.
                    Renal cell cancer (also called kidney cancer or renal adenocarcinoma) is a disease in which malignant (can